In [ ]:
# Dependency setup — safe to run with "Run All"; no manual kernel restart required.
# Fixes the PyTorch CUDA 13.0 / TorchAudio CUDA 12.8 mismatch without replacing PyTorch.

import importlib
import importlib.metadata as metadata
import subprocess
import sys
from pathlib import Path


def _is_installed(package_name):
    try:
        metadata.version(package_name)
        return True
    except metadata.PackageNotFoundError:
        return False


def _purge_cached_modules(prefixes):
    # If a previous failed run partially imported these libraries, discard their
    # cached Python modules so the next imports reflect the installed files.
    for module_name in list(sys.modules):
        if any(module_name == prefix or module_name.startswith(prefix + ".") for prefix in prefixes):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()


# This image-only worker does not need TorchAudio. Removing the incompatible
# optional package makes Transformers skip its audio backend. Do NOT reinstall
# or upgrade torch, torchvision, triton, or CUDA packages here.
packages_to_remove = [name for name in ("torchaudio", "torchao") if _is_installed(name)]
if packages_to_remove:
    subprocess.check_call([
        sys.executable, "-m", "pip", "uninstall", "-y", *packages_to_remove
    ])
    print("Removed optional packages:", ", ".join(packages_to_remove))
else:
    print("No incompatible optional audio/torchao package detected.")

# Clear stale imports from an earlier failed attempt, allowing the current
# notebook session to continue without a manual restart.
_HF_MODULES = ("torchaudio", "torchao", "transformers", "diffusers", "accelerate", "peft")
_purge_cached_modules(_HF_MODULES)


def _distribution_has_file(distribution_name, relative_path):
    try:
        distribution = metadata.distribution(distribution_name)
        return Path(distribution.locate_file(relative_path)).is_file()
    except metadata.PackageNotFoundError:
        return False


# Upgrade Hugging Face packages only if they are missing or too old for the
# Flux.2 Klein/Qwen3 pipeline. This avoids downloading/upgrading packages on
# every run. PyTorch and its CUDA build are deliberately left untouched.
needs_hf_update = not (
    _distribution_has_file("diffusers", "diffusers/pipelines/flux2/pipeline_flux2_klein.py")
    and _distribution_has_file("transformers", "transformers/models/qwen3/modeling_qwen3.py")
    and all(_is_installed(name) for name in ("accelerate", "peft", "requests", "pillow"))
)

if needs_hf_update:
    print("Installing/updating required Hugging Face packages; PyTorch will not be upgraded explicitly...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "diffusers", "transformers", "accelerate", "peft", "requests", "pillow"
    ])
    _purge_cached_modules(_HF_MODULES)
else:
    print("Required Flux.2 Klein dependencies are already installed; skipping pip upgrade.")

for package in ("torch", "torchvision", "triton", "diffusers", "transformers", "accelerate", "peft", "requests", "pillow"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: MISSING")

print("\nDependency setup complete. Continue to the next cell; no manual restart is required.")


In [ ]:
import os
import sys
import gc
import time
import base64
import io
import threading
import warnings
from pathlib import Path
from typing import Optional, Union, Dict, Any

# Enforce explicit PyTorch CUDA memory segmenting before initialization
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("HF_HOME", "/tmp/hf_home")

def hide_torchao() -> None:
    """Purges outdated Kaggle torchao modules to prevent FqnToConfig import errors."""
    for name in [m for m in sys.modules if m == "torchao" or m.startswith("torchao.")]:
        del sys.modules[name]
    sys.modules["torchao"] = None
    for name in [m for m in sys.modules if m.split(".")[0] in ("diffusers", "peft")]:
        del sys.modules[name]

hide_torchao()

import numpy as np
import torch
from PIL import Image, ImageOps
import diffusers
from diffusers import Flux2KleinPipeline

warnings.filterwarnings("ignore", category=FutureWarning)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Configure Kaggle Session Accelerator to 'GPU T4 x1'.")

OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def vram_report() -> str:
    free, total = torch.cuda.mem_get_info()
    gb = 1024 ** 3
    return f"VRAM | Allocated: {torch.cuda.memory_allocated() / gb:.2f} GB | Free: {free / gb:.2f} GB / {total / gb:.2f} GB"

def flush_memory(verbose: bool = False) -> None:
    """Executes multi-stage garbage collection and CUDA cache clearing."""
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    gc.collect()
    if verbose:
        print(vram_report())

class FluxStoryEngine:
    """Unified Text-to-Image / Image-to-Image engine optimized for Kaggle T4 GPUs."""

    TXT2IMG = "txt2img"
    IMG2IMG = "img2img"
    DEFAULT_SIZE = (1024, 576)

    def __init__(self, model_id: str = "black-forest-labs/FLUX.2-klein-4B",
                 output_dir: Union[str, Path] = OUTPUT_DIR):
        self.model_id = model_id
        # Select a dtype supported by the selected Kaggle GPU (T4 typically uses float16).
        try:
            self.dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        except (AttributeError, RuntimeError):
            self.dtype = torch.float16
        self.offload_mode = "model"
        self.num_inference_steps = 4
        self.guidance_scale = 0.0
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.pipe = None
        self.active_mode: Optional[str] = None
        self.lock = threading.Lock()

    def _load_pipeline(self, mode: str) -> None:
        flush_memory()
        print(f"[Engine] Loading Pipeline | Mode: {mode} | Dtype: {self.dtype} | Offload: {self.offload_mode}")
        pipe = Flux2KleinPipeline.from_pretrained(
            self.model_id,
            torch_dtype=self.dtype,
            low_cpu_mem_usage=True
        )
        if self.offload_mode == "sequential":
            pipe.enable_sequential_cpu_offload()
        else:
            pipe.enable_model_cpu_offload()

        for fn in ("enable_tiling", "enable_slicing"):
            try:
                getattr(pipe.vae, fn)()
            except (AttributeError, NotImplementedError):
                pass
        self.pipe, self.active_mode = pipe, mode
        print(f"[Engine] Pipeline Ready. {vram_report()}")

    def _teardown(self) -> None:
        if self.pipe is None:
            return
        print(f"[Engine] Tearing down active pipeline...")
        try:
            self.pipe.remove_all_hooks()
        except Exception:
            pass
        del self.pipe
        self.pipe, self.active_mode = None, None
        flush_memory(verbose=True)

    def _ensure_mode(self, mode: str) -> None:
        if self.pipe is None:
            self._load_pipeline(mode)
        elif mode != self.active_mode:
            self.active_mode = mode

    @staticmethod
    def _snap16(v: int) -> int:
        return max(16, (int(v) // 16) * 16)

    @staticmethod
    def _is_degenerate(img: Image.Image) -> bool:
        arr = np.asarray(img.resize((64, 64)), dtype=np.float32)
        return (not np.isfinite(arr).all()) or float(arr.std()) < 2.0

    def _run(self, mode: str, call_kwargs: dict, seed: Optional[int]) -> Image.Image:
        for attempt in range(4):
            self._ensure_mode(mode)
            generator = torch.Generator(device="cpu").manual_seed(seed) if seed is not None else None
            oom = False
            try:
                with torch.inference_mode():
                    out = self.pipe(generator=generator, **call_kwargs).images[0]
            except torch.cuda.OutOfMemoryError:
                oom = True
            finally:
                flush_memory()

            if oom:
                if self.offload_mode != "sequential":
                    print("[Engine] OOM Detected. Fallback to Sequential CPU Offload...")
                    self.offload_mode = "sequential"
                    self._teardown()
                    continue
                raise RuntimeError("CUDA OOM under sequential offload. Reduce resolution parameters.")

            if self._is_degenerate(out) and self.dtype == torch.bfloat16:
                print("[Engine] Degenerate/NaN Frame Detected under bfloat16. Switching to float16...")
                self.dtype = torch.float16
                self._teardown()
                continue
            return out
        raise RuntimeError("Generation pipeline failed after maximum fallback attempts.")

    def generate(self, prompt: str, image: Optional[Image.Image] = None,
                 width: Optional[int] = None, height: Optional[int] = None,
                 strength: float = 1.0, seed: Optional[int] = None,
                 save_name: Optional[str] = None) -> Image.Image:
        with self.lock:
            if not prompt or not prompt.strip():
                raise ValueError("Prompt cannot be empty.")
            
            mode = self.IMG2IMG if image is not None else self.TXT2IMG
            if mode == self.IMG2IMG:
                width = self._snap16(width or image.width)
                height = self._snap16(height or image.height)
                source = ImageOps.fit(image.convert("RGB"), (width, height), method=Image.LANCZOS)
            else:
                width = self._snap16(width or self.DEFAULT_SIZE[0])
                height = self._snap16(height or self.DEFAULT_SIZE[1])

            call_kwargs = dict(
                prompt=prompt, height=height, width=width,
                num_inference_steps=self.num_inference_steps,
                guidance_scale=self.guidance_scale
            )
            if mode == self.IMG2IMG:
                call_kwargs["image"] = source

            result = self._run(mode, call_kwargs, seed)
            if mode == self.IMG2IMG and strength < 1.0:
                result = Image.blend(source, result, alpha=strength)

            file_path = self.output_dir / f"{save_name or f'{mode}_{int(time.time())}'}.png"
            result.save(file_path)
            return result


# FluxStory — On-Demand Kaggle Batch GPU Worker

## Purpose

This notebook is a **finite batch worker**, not a public HTTP server. Flask owns users, the durable job database, queue order, progress, output storage, and browser polling. When GPU work is waiting, a Flask-side dispatcher can start a private Kaggle notebook run using the documented Kaggle CLI. This notebook claims internal GPU requests from Flask, generates images sequentially, posts the PNG results back, and **exits after a configurable idle window**. The process ending releases the batch execution naturally; this notebook does not keep an ngrok tunnel or permanent FastAPI server alive.

## One-time setup

1. Create a **private** Kaggle notebook/kernel from this notebook. Render-triggered runs use `kaggle kernels push`, which cannot attach UI-managed Kaggle Secrets. Create a **private Kaggle Dataset** with slug `fluxstory-worker-secrets` containing `worker_callback_token.txt` with the same token as Render. The worker reads `/kaggle/input/fluxstory-worker-secrets/worker_callback_token.txt`. Never print the token.
   Render injects the public HTTPS URL and notebook ID into each run. `kernel-metadata.json` attaches the private token dataset to every pushed version.
2. Configure the kernel metadata to enable Internet and a supported GPU (prefer T4 where available). Keep the kernel private.
3. Configure the Flask deployment with Kaggle CLI credentials and the target kernel ID. Never put Kaggle API credentials or the callback token in source code.
4. The Flask implementation must expose the authenticated internal routes described in the Codex prompt: `/internal/gpu/worker/started`, `/internal/gpu/worker/heartbeat`, `/internal/gpu/claim`, `/internal/gpu/requests/{request_id}/complete`, `/internal/gpu/requests/{request_id}/fail`, and `/internal/gpu/worker/exiting`.

## Execution behavior

- The pipeline is loaded only when the first request is claimed.
- One job is generated at a time using the existing `FluxStoryEngine`.
- Empty queue for `WORKER_IDLE_EXIT_SECONDS` causes a normal exit.
- `WORKER_MAX_RUNTIME_SECONDS` bounds a single run; after this limit the worker stops claiming new jobs, finishes the current image, and exits so Flask can dispatch another run for remaining work.
- The notebook never writes to Flask's database directly. It uses authenticated HTTPS callbacks; Flask persists job state and stores image files.

## Runtime compatibility note

The first code cell removes the incompatible optional `torchaudio` package while preserving Kaggle's preinstalled PyTorch/CUDA stack. Restart the kernel after running that cell. The engine uses the matching `black-forest-labs/FLUX.2-klein-4B` checkpoint with `Flux2KleinPipeline` and selects BF16 only when the active GPU reports support; otherwise it uses FP16.

## Important

Kaggle capacity, execution limits, and terms apply. This finite batch pattern is not an uptime guarantee. Verify that your intended automated workload complies with Kaggle's current Acceptable Use Policy before enabling it for a public-facing application. Do not use keepalive tricks or run a permanent tunnel.


In [ ]:
import base64
import io
import json
import logging
import os
import re
import threading
import time
import uuid
from typing import Optional, Dict, Any

import requests
from PIL import Image, UnidentifiedImageError

logger = logging.getLogger("fluxstory.kaggle_batch_worker")
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")


def _read_kaggle_secret(name: str) -> Optional[str]:
    # UI-attached Kaggle Secrets may be unavailable in kernels-push runs.
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value and value.strip():
            return value.strip()
    except Exception:
        pass
    if name == "WORKER_CALLBACK_TOKEN":
        token_file = Path(os.environ.get(
            "WORKER_CALLBACK_TOKEN_FILE",
            "/kaggle/input/fluxstory-worker-secrets/worker_callback_token.txt",
        ))
        try:
            value = token_file.read_text(encoding="utf-8").strip()
            return value or None
        except OSError:
            return None
    return None


def _config_value(name: str, default: Optional[str] = None) -> Optional[str]:
    return os.environ.get(name) or _read_kaggle_secret(name) or default


FLASK_WORKER_BASE_URL = (_config_value("FLASK_WORKER_BASE_URL") or "").rstrip("/")
WORKER_CALLBACK_TOKEN = _config_value("WORKER_CALLBACK_TOKEN") or ""
IDLE_EXIT_SECONDS = int(_config_value("WORKER_IDLE_EXIT_SECONDS", "45"))
MAX_RUNTIME_SECONDS = int(_config_value("WORKER_MAX_RUNTIME_SECONDS", str(4 * 60 * 60)))
POLL_INTERVAL_SECONDS = float(_config_value("WORKER_POLL_INTERVAL_SECONDS", "2"))
HTTP_TIMEOUT_SECONDS = int(_config_value("WORKER_HTTP_TIMEOUT_SECONDS", "45"))
MAX_PROMPT_CHARS = 12000
MAX_IMAGE_B64_CHARS = 20_000_000
WORKER_RUN_ID = str(uuid.uuid4())

if not FLASK_WORKER_BASE_URL.startswith("https://"):
    raise RuntimeError("FLASK_WORKER_BASE_URL must be injected as a public HTTPS URL before the Kaggle push.")
if not WORKER_CALLBACK_TOKEN or len(WORKER_CALLBACK_TOKEN) < 32:
    raise RuntimeError("WORKER_CALLBACK_TOKEN is unavailable or too short. Attach the private fluxstory-worker-secrets dataset with worker_callback_token.txt and keep the same token in Flask/Render.")
if IDLE_EXIT_SECONDS < 10 or IDLE_EXIT_SECONDS > 600:
    raise RuntimeError("WORKER_IDLE_EXIT_SECONDS must be between 10 and 600 seconds.")
if MAX_RUNTIME_SECONDS < 300:
    raise RuntimeError("WORKER_MAX_RUNTIME_SECONDS must be at least 300 seconds.")


def _headers() -> Dict[str, str]:
    return {
        "X-Worker-Token": WORKER_CALLBACK_TOKEN,
        "X-Worker-Run-ID": WORKER_RUN_ID,
        "User-Agent": "FluxStory-Kaggle-Batch-Worker/1.0",
    }


def _post(path: str, payload: Dict[str, Any], timeout: Optional[int] = None, retries: int = 3) -> Dict[str, Any]:
    """POST JSON with bounded retries. Flask endpoints must be idempotent by run/request ID."""
    url = f"{FLASK_WORKER_BASE_URL}{path}"
    last_error = None
    for attempt in range(retries):
        try:
            response = requests.post(
                url,
                json=payload,
                headers=_headers(),
                timeout=(10, timeout or HTTP_TIMEOUT_SECONDS),
            )
            if response.status_code in (408, 429, 500, 502, 503, 504) and attempt + 1 < retries:
                retry_after = response.headers.get("Retry-After")
                try:
                    delay = min(15.0, max(1.0, float(retry_after))) if retry_after else 2 ** attempt
                except ValueError:
                    delay = 2 ** attempt
                time.sleep(delay)
                continue
            response.raise_for_status()
            if not response.content:
                return {}
            data = response.json()
            if not isinstance(data, dict):
                raise RuntimeError(f"Invalid JSON object returned by Flask route {path}")
            return data
        except (requests.Timeout, requests.ConnectionError, requests.HTTPError, ValueError, RuntimeError) as exc:
            last_error = exc
            # Do not retry authorization/configuration/validation failures.
            if isinstance(exc, requests.HTTPError):
                status_code = exc.response.status_code if exc.response is not None else None
                if status_code not in (408, 429, 500, 502, 503, 504):
                    raise
            if attempt + 1 < retries:
                time.sleep(min(8.0, 2 ** attempt))
    raise RuntimeError(f"Could not call Flask internal worker endpoint {path}: {type(last_error).__name__}") from last_error


def _decode_input_image(image_b64: Optional[str]) -> Optional[Image.Image]:
    if not image_b64:
        return None
    encoded = str(image_b64).strip()
    if encoded.startswith("data:"):
        if "," not in encoded:
            raise ValueError("Invalid input image data URI")
        encoded = encoded.split(",", 1)[1]
    if len(encoded) > MAX_IMAGE_B64_CHARS:
        raise ValueError("Input image payload exceeds the configured limit")
    try:
        raw = base64.b64decode(encoded, validate=True)
        with Image.open(io.BytesIO(raw)) as opened:
            opened.verify()
        with Image.open(io.BytesIO(raw)) as opened:
            if opened.width * opened.height > 16_000_000:
                raise ValueError("Input image dimensions exceed the limit")
            return opened.convert("RGB").copy()
    except (ValueError, UnidentifiedImageError, OSError) as exc:
        raise ValueError("image_b64 must contain a valid encoded image") from exc


def _safe_save_name(raw: Optional[str]) -> str:
    name = re.sub(r"[^A-Za-z0-9_-]", "_", str(raw or "image")).strip("_")[:70] or "image"
    return f"{name}_{uuid.uuid4().hex[:8]}"


engine_instance = FluxStoryEngine(output_dir=OUTPUT_DIR)
_started_monotonic = time.monotonic()
_last_heartbeat = 0.0
_heartbeat_stop = threading.Event()
_heartbeat_thread = None


def _heartbeat_loop() -> None:
    """Keep Flask aware of long GPU generations without interfering with inference."""
    session_start = time.time()
    while not _heartbeat_stop.wait(15):
        try:
            _post("/internal/gpu/worker/heartbeat", {
                "worker_run_id": WORKER_RUN_ID,
                "state": "processing" if engine_instance.pipe is not None else "idle",
                "gpu_available": bool(torch.cuda.is_available()),
                "engine_loaded": engine_instance.pipe is not None,
                "elapsed_seconds": int(time.monotonic() - _started_monotonic),
                "heartbeat_at": time.time(),
            }, timeout=20, retries=1)
        except Exception as exc:
            logger.warning("Heartbeat delivery failed: %s", type(exc).__name__)


def _validate_request(data: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(data, dict):
        raise ValueError("Claim response request must be an object")
    request_id = str(data.get("request_id") or "").strip()
    prompt = str(data.get("prompt") or "").strip()
    if not request_id:
        raise ValueError("GPU request is missing request_id")
    if not prompt or len(prompt) > MAX_PROMPT_CHARS:
        raise ValueError("Prompt is empty or exceeds maximum length")
    width = int(data.get("width", 1024))
    height = int(data.get("height", 576))
    if width < 16 or height < 16 or width > 2048 or height > 2048 or width % 16 or height % 16 or width * height > 4_194_304:
        raise ValueError("Image dimensions must be within 16..1024")
    seed = data.get("seed", 42)
    if seed is not None:
        seed = int(seed)
        if seed < 0 or seed > 2**63 - 1:
            raise ValueError("Seed is out of range")
    strength = float(data.get("strength", 1.0))
    if not 0.0 <= strength <= 1.0:
        raise ValueError("Strength must be between 0 and 1")
    image_b64 = data.get("image_b64")
    if image_b64 is not None and not isinstance(image_b64, str):
        raise ValueError("image_b64 must be a string")
    return {
        "request_id": request_id,
        "prompt": prompt,
        "width": width,
        "height": height,
        "seed": seed,
        "strength": strength,
        "image_b64": image_b64,
        "save_name": _safe_save_name(data.get("save_name") or request_id),
    }


def _process_request(raw_request: Dict[str, Any]) -> None:
    request = _validate_request(raw_request)
    request_id = request["request_id"]
    started = time.monotonic()
    try:
        source_image = _decode_input_image(request["image_b64"])
        image = engine_instance.generate(
            prompt=request["prompt"],
            image=source_image,
            width=request["width"],
            height=request["height"],
            strength=request["strength"],
            seed=request["seed"],
            save_name=request["save_name"],
        )
        output = io.BytesIO()
        image.save(output, format="PNG", optimize=True)
        encoded = base64.b64encode(output.getvalue()).decode("ascii")
        _post(f"/internal/gpu/requests/{request_id}/complete", {
            "worker_run_id": WORKER_RUN_ID,
            "image_b64": encoded,
            "width": int(image.width),
            "height": int(image.height),
            "elapsed_seconds": round(time.monotonic() - started, 3),
        }, timeout=120, retries=3)
        logger.info("Completed GPU request %s (%sx%s) in %.1fs", request_id, image.width, image.height, time.monotonic() - started)
    except Exception as exc:
        logger.error("GPU request failed (%s)", type(exc).__name__)
        try:
            _post(f"/internal/gpu/requests/{request_id}/fail", {
                "worker_run_id": WORKER_RUN_ID,
                "error_code": type(exc).__name__,
                "message": "Image generation failed in the Kaggle worker.",
            }, timeout=30, retries=2)
        except Exception as callback_exc:
            logger.error("Could not report GPU request failure (%s)", type(callback_exc).__name__)


exit_reason = "unknown"
try:
    _post("/internal/gpu/worker/started", {
        "worker_run_id": WORKER_RUN_ID,
        "kernel_id": _config_value("KAGGLE_KERNEL_ID", "owner/fluxstory-on-demand-worker"),
        "started_at": time.time(),
        "idle_exit_seconds": IDLE_EXIT_SECONDS,
        "max_runtime_seconds": MAX_RUNTIME_SECONDS,
    }, timeout=30, retries=3)
    _heartbeat_thread = threading.Thread(target=_heartbeat_loop, name="worker-heartbeat", daemon=True)
    _heartbeat_thread.start()

    logger.info("FluxStory batch worker started run_id=%s; idle_exit=%ss; max_runtime=%ss", WORKER_RUN_ID, IDLE_EXIT_SECONDS, MAX_RUNTIME_SECONDS)
    no_work_since = time.monotonic()
    while True:
        elapsed = time.monotonic() - _started_monotonic
        # Leave room for the active image and callbacks before the configured run deadline.
        if elapsed >= MAX_RUNTIME_SECONDS:
            exit_reason = "max_runtime"
            logger.info("Maximum worker runtime reached; leaving remaining jobs queued for the next run")
            break

        try:
            # Flask must make claim idempotent for worker_run_id: if the response is lost,
            # a retry by this same run must return its currently leased request, not another one.
            response = _post("/internal/gpu/claim", {
                "worker_run_id": WORKER_RUN_ID,
                "wait_for_request": False,
            }, timeout=HTTP_TIMEOUT_SECONDS, retries=3)
        except Exception as exc:
            logger.error("Queue claim failed (%s); leaving for Flask lease recovery", type(exc).__name__)
            time.sleep(5)
            # Avoid burning quota in a disconnected worker; let the orchestrator retry later.
            exit_reason = "flask_unreachable"
            break

        request = response.get("request")
        if request:
            # Time spent generating is productive work, not idle time.
            no_work_since = None
            _process_request(request)
            no_work_since = None
            # Image generation is sequential: only this loop calls FluxStoryEngine.generate.
            continue

        if no_work_since is None:
            no_work_since = time.monotonic()
        idle_for = time.monotonic() - no_work_since
        if idle_for >= IDLE_EXIT_SECONDS:
            exit_reason = "idle"
            logger.info("GPU queue empty for %.1fs; exiting batch to release the session", idle_for)
            break
        time.sleep(POLL_INTERVAL_SECONDS)

except Exception as exc:
    exit_reason = "worker_error"
    logger.exception("Batch worker stopped unexpectedly (%s)", type(exc).__name__)
    raise
finally:
    _heartbeat_stop.set()
    if _heartbeat_thread is not None:
        _heartbeat_thread.join(timeout=2)
    try:
        if engine_instance.pipe is not None:
            engine_instance._teardown()
    except Exception as cleanup_exc:
        logger.warning("Engine cleanup warning: %s", type(cleanup_exc).__name__)
    try:
        _post("/internal/gpu/worker/exiting", {
            "worker_run_id": WORKER_RUN_ID,
            "reason": exit_reason,
            "finished_at": time.time(),
            "elapsed_seconds": int(time.monotonic() - _started_monotonic),
        }, timeout=20, retries=2)
    except Exception as callback_exc:
        logger.warning("Could not send worker exit callback: %s", type(callback_exc).__name__)
    logger.info("FluxStory batch worker finished; reason=%s", exit_reason)
